# BIND2 Thermodynamics — Paper Figures

Same model, gas-thermodynamic outputs ($y, T, K, P_e$). Load-only from `tools/paper_cache/` (`thermo_*` artifacts + `field1p.npz`). CV + 1P (SB35 has no thermo truth).

In [ ]:

import sys
sys.path.insert(0, '/mnt/home/mlee1/vdm_bind2/tools/paper_cache')
import os, pickle
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import paper_config as C

CACHE = C.CACHE_DIR
def L(name):
    p = CACHE / name
    if name.endswith('.pkl'):
        return pickle.load(open(p, 'rb'))
    return dict(np.load(p, allow_pickle=False))

FIG_DIR = Path('paper_figures'); FIG_DIR.mkdir(exist_ok=True)
def save_fig(fig, name, ext=('pdf', 'png')):
    for e in ext:
        fig.savefig(FIG_DIR / f'{name}.{e}', dpi=300, bbox_inches='tight')
    print('  saved', name)

try:
    import scienceplots  # noqa: F401
    plt.style.use(['science', 'notebook'])
except Exception:
    pass
# plt.rcParams.update({'font.size': 10, 'font.family': 'serif', 'mathtext.fontset': 'cm',
#                      'figure.dpi': 110, 'savefig.dpi': 300, 'axes.grid': False})

SUITE_COLORS = C.SUITE_COLORS; SUITE_DISPLAY = C.SUITE_DISPLAY
MASS_CH = C.MASS_CHANNELS; CH_DISPLAY = C.CH_DISPLAY
BIN_LABELS = C.MASS_BIN_LABELS; N_BINS = C.N_MASS_BINS
PARAM_LABELS = C.PARAM_LABELS
# Trained-regime restriction: the paper only uses halos with M200c >= 1e13 (the
# training cut). Lower bins exist in the cache but are never plotted.
MIN_LOG_M200 = 13.0
BINS = [b for b in range(N_BINS) if C.MASS_EDGES[b] >= MIN_LOG_M200]
BIN_CMAP = np.zeros((N_BINS, 4))
BIN_CMAP[BINS] = plt.cm.viridis(np.linspace(0.1, 0.9, len(BINS)))
print('Cache :', CACHE)
print('Model :', C.MODEL_TAG, '| suites', {s: 0 for s in C.SUITES})
print('Files :', sorted(p.name for p in CACHE.glob("*.pkl")) + sorted(p.name for p in CACHE.glob("*.npz")))


In [ ]:

THK = C.THERMO_CHANNELS; THD = C.THERMO_DISPLAY


## T1 · Thermo field showcase (truth vs BIND)

Hero 1P halo, from `field1p.npz` (channels 3–6).

In [ ]:

f1 = L('field1p.npz'); j = int(f1['params'][0])
t = f1[f'p{j}_t_hi']; g = f1[f'p{j}_g_hi']    # (7,128,128)
fig, axes = plt.subplots(2, len(THK), figsize=(3*len(THK), 6))
def lg(im): p=im[im>0]; return np.log10(np.clip(im, p.min() if len(p) else 1e-30, None))
for c in range(len(THK)):
    for r,(src,arr) in enumerate([('Truth',t),('BIND',g)]):
        im = arr[3+c]; axes[r,c].imshow(lg(im), cmap='magma'); axes[r,c].set_xticks([]); axes[r,c].set_yticks([])
        if r==0: axes[r,c].set_title(THD[THK[c]])
        if c==0: axes[r,c].set_ylabel(src)
save_fig(fig, 'figT1_thermo_showcase'); plt.show()


## T2/T3 · Per-pixel accuracy + PDF (`thermo_pixels.npz`)

In [ ]:

tp = L('thermo_pixels.npz')
fig, axes = plt.subplots(2, len(THK), figsize=(3.2*len(THK), 6.2))
for c,name in enumerate(THK):
    t = tp[f'{name}_truth']; g = tp[f'{name}_gen']
    ax = axes[0,c]; ax.hexbin(t, g, gridsize=45, bins='log', cmap='viridis'); lo,hi=np.percentile(t,[1,99]); ax.plot([lo,hi],[lo,hi],'r--',lw=1)
    ax.set_title(f'{THD[name]}: bias {np.median(g-t):+.3f} dex'); ax.set_xlabel(r'$\log_{10}$ truth');
    if c==0: ax.set_ylabel(r'$\log_{10}$ BIND')
    ax2 = axes[1,c]; b=np.linspace(min(t.min(),g.min()), max(t.max(),g.max()), 60)
    ax2.hist(t, bins=b, density=True, alpha=0.4, color='tab:gray', label='Truth'); ax2.hist(g, bins=b, density=True, histtype='step', lw=2, color='tab:orange', label='BIND')
    ax2.set_xlabel(r'$\log_{10}$ value');
    if c==0: ax2.set_ylabel('PDF'); ax2.legend()
save_fig(fig, 'figT2_thermo_pixels'); plt.show()


## T4 · Thermo radial profiles by mass bin (`thermo_profiles.pkl`)

In [ ]:

pr = L('thermo_profiles.pkl'); r = pr['r']
fig, axes = plt.subplots(len(THK), 1, figsize=(7, 2.8*len(THK)), sharex=True)
for c, ax in enumerate(axes):
    for b in BINS:
        if b not in pr['by_bin']: continue
        band = pr['by_bin'][b]
        ax.plot(r, band['pctdiff_med'][c], color=BIN_CMAP[b], lw=1.7, label=BIN_LABELS[b] if c==0 else None)
        ax.fill_between(r, band['pctdiff_p16'][c], band['pctdiff_p84'][c], color=BIN_CMAP[b], alpha=0.10)
    ax.axhline(0, color='k', ls='--', lw=0.8); ax.set_xscale('log'); ax.set_ylim(-1,1); ax.set_ylabel(rf'$\Delta$ {THD[THK[c]]}/{THD[THK[c]]}'); ax.grid(which='both', alpha=0.2)
    if c==0: ax.legend(title=r'$\log_{10}M_{200c}$', fontsize=8)
axes[-1].set_xlabel(r'$r$ [Mpc/$h$]')
save_fig(fig, 'figT4_thermo_profiles_by_bin'); plt.show()


## T5 · Scaling relations Y–M, T–M, K–M, P–M (`thermo_scaling.pkl`)

In [ ]:

sc = L('thermo_scaling.pkl')
keys = [('Y','$Y_{200}$'),('Tx','$T_{200}$'),('K','$K_{200}$'),('P','$P_{200}$')]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
for ax,(key,lab) in zip(axes, keys):
    x = sc['logM'];
    for src,cc in [('truth','k'),('gen','tab:orange')]:
        y = sc[f'{src}_{key}']; m=(y>0)&np.isfinite(x);
        ax.scatter(x[m], np.log10(y[m]), s=3, alpha=0.05, color=cc, rasterized=True)
        b=np.linspace(x[m].min(), x[m].max(), 14); bc=0.5*(b[:-1]+b[1:]); idx=np.digitize(x[m],b)-1; yl=np.log10(y[m])
        med=[np.median(yl[idx==i]) if (idx==i).sum()>5 else np.nan for i in range(len(bc))]
        ax.plot(bc, med, 'o-', color=cc, label={'truth':'Truth','gen':'BIND'}[src], mec='k')
    ax.set_xlabel(r'$\log_{10}M_{200c}$'); ax.set_ylabel(rf'$\log_{{10}}$ {lab}'); ax.set_title(lab);
axes[0].legend()
save_fig(fig, 'figT5_scaling_relations'); plt.show()


## T6 · Joint mass + SZ scaling-relation scatter & residual corner plot (`mass_table.pkl` + `thermo_scaling.pkl`)

Per-halo scaling relations on **CV** (fixed fiducial parameters, so residuals are halo-to-halo scatter, not parameter response), trained regime $M_{200c}\ge 10^{13}$, **clean halos only**: the eval maps are full-box-depth (50 Mpc/$h$) projections, so an R200c aperture also sums every *projected* neighbor along the LOS — an additive, strictly-positive boost that lifts $M_\star$, $M_{\rm gas}$ and $Y$ coherently and manufactures a spurious $+1\sigma$ cross-relation ridge (diagnosed in figT6c below). Halos with external catalog mass $>0.2\,M_{200}$ inside the aperture (~6%) are dropped before fitting. Residuals are w.r.t. each population's **own** OLS power-law fit; the corner plot classifies halos into the $\pm 1\sigma$ tails of the *column* relation and tests whether BIND (dotted) reproduces the truth's (solid) cross-relation outlier structure. figT6d isolates the physics that remains after cleaning.

In [ ]:

from scipy import stats

# Per-halo join: mass_table (R200-aperture masses) + thermo_scaling (Y, Tx) come
# from the same catalogs in the same row order -> align by (suite, sim, row) and
# verify with the shared log-mass column (guards against silent scrambling).
mt = L('mass_table.pkl'); th = L('thermo_scaling.pkl')
mt = mt.assign(row=mt.groupby(['suite', 'sim_id']).cumcount())
th = th.assign(row=th.groupby(['suite', 'sim_id']).cumcount())
tab = mt.merge(th, on=['suite', 'sim_id', 'row'], suffixes=('', '_th'))
assert np.allclose(tab['log_m200c'], tab['logM']), 'mass/thermo halo rows misaligned'

MASS_MIN_LOG = np.log10(C.PARAM_RESPONSE_MASS_MIN)
tab = tab[(tab.suite == 'CV') & (tab.log_m200c >= MASS_MIN_LOG)].reset_index(drop=True)
M200 = 10.0 ** tab['log_m200c'].to_numpy()

# R200-aperture contamination by projected neighbors: the maps are full-box-depth
# projections, so the aperture sums every halo along the LOS. External mass adds
# coherently (and strictly positively) to Mstar, Mgas and Y -> flag before fitting.
recs = {r['sim_id']: r for r in C.discover_sims(('CV',))}
contam = np.full(len(tab), np.nan)          # external catalog mass / own M200
for sim_id, sub in tab.groupby('sim_id'):
    cat = C.load_catalog(recs[sim_id])
    cen2 = np.asarray(cat['centers']); m_all = np.asarray(cat['masses'], float)
    r200 = np.asarray(cat['r200s'], float)
    d = np.abs(cen2[:, None, :] - cen2[None, :, :])
    d = np.minimum(d, C.BOX_SIZE - d)                       # periodic
    dproj = np.hypot(d[..., 0], d[..., 1])
    rows = sub['row'].to_numpy()
    assert np.allclose(np.log10(m_all[rows]), sub['log_m200c']), sim_id
    for i, ri in zip(sub.index, rows):
        ins = dproj[ri] < r200[ri]; ins[ri] = False
        contam[i] = m_all[ins].sum() / m_all[ri]
CONTAM_MAX = 0.2
clean = contam < CONTAM_MAX
print(f'{len(tab)} CV halos with log10 M200c >= {MASS_MIN_LOG:.0f}; removing '
      f'{(~clean).sum()} aperture-contaminated (M_ext > {CONTAM_MAX} M200) '
      f'-> {clean.sum()} clean')

def fit_mean_relation(x, y, fit_on=None):
    """OLS log10(y) = a*log10(x) + b fitted on the `fit_on` subset; residuals are
    returned for ALL valid halos (so contaminated halos can still be plotted
    against the clean relation in figT6c). sigma = clean-sample scatter."""
    valid = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    fmask = valid if fit_on is None else (valid & fit_on)
    a, b, *_ = stats.linregress(np.log10(x[fmask]), np.log10(y[fmask]))
    res = np.full(len(x), np.nan)
    res[valid] = np.log10(y[valid]) - (a * np.log10(x[valid]) + b)
    return a, b, np.nanstd(res[fmask]), valid, res

g = lambda k: tab[k].to_numpy()
# key -> (x_truth, y_truth, x_bind, y_bind, xlabel, ylabel, title)
RELATIONS = {
    'Mgas-Mstar': (g('truth_Stars_rvir'), g('truth_Gas_rvir'), g('gen_Stars_rvir'), g('gen_Gas_rvir'),
                   r'$M_\star$', r'$M_{\rm gas}$', r'$M_{\rm gas}-M_\star$'),
    'Mdm-Mstar':  (g('truth_Stars_rvir'), g('truth_DM_hydro_rvir'), g('gen_Stars_rvir'), g('gen_DM_hydro_rvir'),
                   r'$M_\star$', r'$M_{\rm dm}$', r'$M_{\rm dm}-M_\star$'),
    'SHMR':       (M200, g('truth_Stars_rvir'), M200, g('gen_Stars_rvir'),
                   r'$M_{200c}$', r'$M_\star$', 'SHMR'),
    'BaryonFrac': (M200, g('truth_Gas_rvir') + g('truth_Stars_rvir'), M200, g('gen_Gas_rvir') + g('gen_Stars_rvir'),
                   r'$M_{200c}$', r'$M_{\rm baryon}$', r'$(M_{\rm gas}+M_\star)-M_{200}$'),
    'Y-M':        (M200, g('truth_Y'), M200, g('gen_Y'),
                   r'$M_{200c}$', r'$Y_{200}$', r'$Y-M_{200}$'),
    'Y-T':        (g('truth_Tx'), g('truth_Y'), g('gen_Tx'), g('gen_Y'),
                   r'$T_{200}$', r'$Y_{200}$', r'$Y-T$'),
}
# auxiliary relations for the physics figure (figT6d); not in the grid/corner
AUX = {
    'Mgas-M': (M200, g('truth_Gas_rvir'), M200, g('gen_Gas_rvir'),
               r'$M_{200c}$', r'$M_{\rm gas}$', r'$M_{\rm gas}-M_{200}$'),
    'T-M':    (M200, g('truth_Tx'), M200, g('gen_Tx'),
               r'$M_{200c}$', r'$T_{200}$', r'$T-M_{200}$'),
}
fits = {'Truth': {}, 'BIND': {}}
print(f"{'relation':>12s}   truth a/sigma      BIND a/sigma   (clean-sample fits)")
for key, (xt, yt, xg, yg, xl, yl, ti) in {**RELATIONS, **AUX}.items():
    for src, x, y in [('Truth', xt, yt), ('BIND', xg, yg)]:
        a, b, s, v, r = fit_mean_relation(x, y, fit_on=clean)
        fits[src][key] = dict(alpha=a, beta=b, sigma=s, valid=v, mask=v & clean,
                              resid=r, x=x, y=y, xlabel=xl, ylabel=yl, title=ti)
    ft, fg = fits['Truth'][key], fits['BIND'][key]
    print(f"{key:>12s}   {ft['alpha']:+.2f} / {ft['sigma']:.2f}     {fg['alpha']:+.2f} / {fg['sigma']:.2f}")


In [ ]:

keys = list(RELATIONS)
VLIM = 0.25
fig, axes = plt.subplots(2, len(keys), figsize=(3.3 * len(keys), 6.4), constrained_layout=True)
for row, src in enumerate(['Truth', 'BIND']):
    for col, key in enumerate(keys):
        ax = axes[row, col]; fd = fits[src][key]
        lx = np.log10(fd['x'][fd['mask']]); ly = np.log10(fd['y'][fd['mask']])
        sc = ax.scatter(lx, ly, c=fd['resid'][fd['mask']], cmap='RdBu_r',
                        vmin=-VLIM, vmax=VLIM, s=14, rasterized=True, zorder=3)
        xl_ = np.linspace(lx.min(), lx.max(), 50)
        ax.plot(xl_, fd['alpha'] * xl_ + fd['beta'], 'k--', lw=1.2)
        ax.fill_between(xl_, fd['alpha'] * xl_ + fd['beta'] - fd['sigma'],
                        fd['alpha'] * xl_ + fd['beta'] + fd['sigma'], color='k', alpha=0.10)
        ax.text(0.04, 0.96, f"{src}\n" + rf"$\alpha={fd['alpha']:.2f},\ \sigma={fd['sigma']:.2f}$ dex",
                transform=ax.transAxes, va='top', fontsize=8, color='0.35')
        if row == 0:
            ax.set_title(fd['title'])
        if row == 1:
            ax.set_xlabel(r'$\log_{10}($' + fd['xlabel'] + r'$)$')
        ax.set_ylabel(r'$\log_{10}($' + fd['ylabel'] + r'$)$')
cb = fig.colorbar(sc, ax=axes, fraction=0.015, pad=0.01)
cb.set_label('residual [dex]')
save_fig(fig, 'figT6a_scaling_scatter'); plt.show()


In [ ]:

from scipy.stats import gaussian_kde

CORNER_LABELS = {'Mgas-Mstar': r'$r_{M_{\rm gas}-M_\star}$', 'Mdm-Mstar': r'$r_{M_{\rm dm}-M_\star}$',
                 'SHMR': r'$r_{\rm SHMR}$', 'BaryonFrac': r'$r_{M_{\rm bar}-M_{200}}$',
                 'Y-M': r'$r_{Y-M}$', 'Y-T': r'$r_{Y-T}$'}
ckeys = list(RELATIONS)
both = np.ones(len(tab), bool)
for k in ckeys:
    both &= fits['Truth'][k]['mask'] & fits['BIND'][k]['mask']
R = {src: np.stack([fits[src][k]['resid'][both] for k in ckeys], 1) for src in ('Truth', 'BIND')}
sig = {src: np.array([fits[src][k]['sigma'] for k in ckeys]) for src in ('Truth', 'BIND')}
nrel = len(ckeys)
rng_lim = []
for j in range(nrel):
    v = np.r_[R['Truth'][:, j], R['BIND'][:, j]]
    lo, hi = np.nanpercentile(v, 1), np.nanpercentile(v, 99)
    pad = 0.15 * (hi - lo); rng_lim.append((lo - pad, hi + pad))

def kde_contour(ax, x, y, color, ls):
    """1/2/3-sigma-enclosure KDE contours of the (x, y) residual sub-population."""
    if len(x) < 10 or np.ptp(x) < 1e-10 or np.ptp(y) < 1e-10:
        return
    kde = gaussian_kde(np.vstack([x, y]), bw_method='scott')
    xg, yg = np.mgrid[x.min()-0.2*np.ptp(x):x.max()+0.2*np.ptp(x):80j,
                      y.min()-0.2*np.ptp(y):y.max()+0.2*np.ptp(y):80j]
    z = kde(np.vstack([xg.ravel(), yg.ravel()])).reshape(xg.shape)
    zs = np.sort(z.ravel())[::-1]; cf = np.cumsum(zs) / zs.sum()
    lv = sorted({float(zs[min(np.searchsorted(cf, f), len(zs)-1)]) for f in (0.6827, 0.9545, 0.9973)})
    ax.contour(xg, yg, z, levels=lv, colors=[color], linewidths=1.2, linestyles=ls, alpha=0.85)

fig, axes = plt.subplots(nrel, nrel, figsize=(2.4 * nrel, 2.4 * nrel), constrained_layout=True)
for i in range(nrel):
    for j in range(nrel):
        ax = axes[i, j]
        if j > i:
            ax.set_visible(False); continue
        tails = {src: (R[src][:, j] > sig[src][j], R[src][:, j] < -sig[src][j]) for src in R}
        if i == j:
            bins = np.linspace(*rng_lim[i], 35)
            ax.hist(R['Truth'][tails['Truth'][0], i], bins=bins, color='tomato', density=True, alpha=0.7)
            ax.hist(R['Truth'][tails['Truth'][1], i], bins=bins, color='steelblue', density=True, alpha=0.7)
            ax.hist(R['BIND'][tails['BIND'][0], i], bins=bins, color='tomato', density=True, histtype='step', ls=':', lw=1.8)
            ax.hist(R['BIND'][tails['BIND'][1], i], bins=bins, color='steelblue', density=True, histtype='step', ls=':', lw=1.8)
            ax.axvline(sig['Truth'][i], c='tomato', lw=0.8, ls=':', alpha=0.6)
            ax.axvline(-sig['Truth'][i], c='steelblue', lw=0.8, ls=':', alpha=0.6)
            ax.axvline(0, c='k', lw=0.7, ls='--', alpha=0.4)
            ax.set_title(CORNER_LABELS[ckeys[i]], fontsize=10, pad=2)
        else:
            kde_contour(ax, R['Truth'][tails['Truth'][0], j], R['Truth'][tails['Truth'][0], i], 'tomato', '-')
            kde_contour(ax, R['Truth'][tails['Truth'][1], j], R['Truth'][tails['Truth'][1], i], 'steelblue', '-')
            kde_contour(ax, R['BIND'][tails['BIND'][0], j], R['BIND'][tails['BIND'][0], i], 'tomato', ':')
            kde_contour(ax, R['BIND'][tails['BIND'][1], j], R['BIND'][tails['BIND'][1], i], 'steelblue', ':')
            ax.axhline(0, c='k', lw=0.4, ls='--', alpha=0.3); ax.axvline(0, c='k', lw=0.4, ls='--', alpha=0.3)
            ax.set_ylim(rng_lim[i])
        ax.set_xlim(rng_lim[j])
        if j == 0 and i > 0:
            ax.set_ylabel(CORNER_LABELS[ckeys[i]] + '\n[dex]')
        elif j > 0:
            ax.set_yticklabels([])
        if i == nrel - 1:
            ax.set_xlabel(CORNER_LABELS[ckeys[j]] + '\n[dex]')
        else:
            ax.set_xticklabels([])
fig.legend(handles=[Line2D([0], [0], color=c, ls=ls, lw=1.6, label=lb) for c, ls, lb in
                    [('tomato', '-', r'Truth $>+1\sigma$'), ('steelblue', '-', r'Truth $<-1\sigma$'),
                     ('tomato', ':', r'BIND $>+1\sigma$'), ('steelblue', ':', r'BIND $<-1\sigma$')]],
           loc='upper right', frameon=True, fontsize=11)
save_fig(fig, 'figT6b_residual_corner'); plt.show()


### T6c · Why cleaning is necessary — the $+1\sigma$ ridge is aperture contamination

$r_{\rm SHMR}$ vs $r_{Y-M}$ for **all** halos (residuals w.r.t. the clean-sample fits), colored by the projected external catalog mass inside the R200c aperture. Joint positive outliers have median $M_{\rm ext}\approx M_{200}$ (half have a *more massive* projected companion), and only ~16% of the contaminating mass is 3D-associated ($|\Delta{\rm LOS}|<5$ Mpc) — mostly chance superposition through the full-depth projection. There is no 'negative neighbor', so only the positive tail is affected; BIND inherits the ridge because the neighbor is visible in its DMO conditioning patch.

In [ ]:

from scipy.stats import spearmanr

rS = fits['Truth']['SHMR']; rY = fits['Truth']['Y-M']
mall = rS['valid'] & rY['valid']
def _rho(sel):
    return spearmanr(rS['resid'][sel], rY['resid'][sel]).statistic
hi_all = mall & (rS['resid'] > rS['sigma'])
lo_all = mall & (rS['resid'] < -rS['sigma'])
print(f"rho(r_SHMR, r_Y-M) +1s tail: all={_rho(hi_all):+.2f}  clean={_rho(hi_all & clean):+.2f}  "
      f"contaminated={_rho(hi_all & ~clean):+.2f};  -1s tail: {_rho(lo_all):+.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5), constrained_layout=True, sharex=True, sharey=True)
sc = axes[0].scatter(rS['resid'][mall], rY['resid'][mall], c=np.log10(1 + contam[mall]),
                     cmap='inferno_r', vmin=0, vmax=1.0, s=18)
fig.colorbar(sc, ax=axes[0], fraction=0.05).set_label(r'$\log_{10}(1+M_{\rm ext}/M_{200})$ in aperture')
axes[0].set_title('all halos, colored by projected\nneighbor mass in the R200c aperture')
axes[1].scatter(rS['resid'][mall & clean], rY['resid'][mall & clean], s=12, c='0.55',
                alpha=0.5, label='clean')
axes[1].scatter(rS['resid'][mall & ~clean], rY['resid'][mall & ~clean], s=18, c='crimson',
                alpha=0.8, label=rf'contaminated ($M_{{\rm ext}}>{CONTAM_MAX}\,M_{{200}}$)')
axes[1].set_title(rf'$\rho_{{+1\sigma}}$: all $={_rho(hi_all):+.2f}$, '
                  rf'clean $={_rho(hi_all & clean):+.2f}$'
                  '\n' rf'$\rho_{{-1\sigma}} = {_rho(lo_all):+.2f}$')
axes[1].legend(fontsize=9, loc='lower right')
for ax in axes:
    ax.axvline(rS['sigma'], c='tomato', lw=0.9, ls=':')
    ax.axvline(-rS['sigma'], c='steelblue', lw=0.9, ls=':')
    ax.axhline(0, c='k', lw=0.5, ls='--', alpha=0.4); ax.axvline(0, c='k', lw=0.5, ls='--', alpha=0.4)
    ax.set_xlabel(r'$r_{\rm SHMR}$ [dex]')
axes[0].set_ylabel(r'$r_{Y-M}$ [dex]')
save_fig(fig, 'figT6c_aperture_contamination'); plt.show()


### T6d · The physics that survives cleaning: baryon-rich halos, mediated by gas mass

In the clean sample the $+1\sigma$ SHMR halos are still SZ-bright — because they are **gas-rich** at fixed $M_{200}$ (left; the partial correlation of $r_{\rm SHMR}$ with $r_Y$ at fixed $r_{M_{\rm gas}}$ nearly vanishes, and $r_Y$ tracks $r_{M_{\rm gas}}$ tightly, middle). Star-**poor** halos are *not* gas-poor: downward scatter comes from channel-specific suppression (star-formation inefficiency vs feedback gas ejection), so the $-1\sigma$ tails decorrelate (right). BIND (hatched) reproduces the full tail-conditional structure.

In [ ]:

from scipy.stats import spearmanr

def tail_rho(src, xkey, ykey, side):
    fx, fy = fits[src][xkey], fits[src][ykey]
    m = fx['mask'] & fy['mask']
    m &= (fx['resid'] > fx['sigma']) if side == '+' else (fx['resid'] < -fx['sigma'])
    return spearmanr(fx['resid'][m], fy['resid'][m]).statistic, int(m.sum())

rS, rG, rY = (fits['Truth'][k] for k in ('SHMR', 'Mgas-M', 'Y-M'))
mm = rS['mask'] & rG['mask'] & rY['mask']
hi = mm & (rS['resid'] > rS['sigma']); lo = mm & (rS['resid'] < -rS['sigma'])

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.6), constrained_layout=True)

# (a) star-rich halos are gas-rich; star-poor halos are NOT gas-poor
ax = axes[0]
ax.scatter(rS['resid'][mm & ~hi & ~lo], rG['resid'][mm & ~hi & ~lo], s=8, c='0.7', alpha=0.4)
ax.scatter(rS['resid'][hi], rG['resid'][hi], s=16, c='tomato',
           label=rf'$+1\sigma$: $\rho={tail_rho("Truth", "SHMR", "Mgas-M", "+")[0]:+.2f}$')
ax.scatter(rS['resid'][lo], rG['resid'][lo], s=16, c='steelblue',
           label=rf'$-1\sigma$: $\rho={tail_rho("Truth", "SHMR", "Mgas-M", "-")[0]:+.2f}$')
ax.axhline(0, c='k', lw=0.5, ls='--', alpha=0.4); ax.axvline(0, c='k', lw=0.5, ls='--', alpha=0.4)
ax.set_xlabel(r'$r_{\rm SHMR}$ [dex]'); ax.set_ylabel(r'$r_{M_{\rm gas}-M}$ [dex]')
ax.legend(fontsize=9); ax.set_title('clean sample (Truth)')

# (b) the Y excess is carried by gas mass
ax = axes[1]
sc = ax.scatter(rG['resid'][mm], rY['resid'][mm], c=rS['resid'][mm], cmap='RdBu_r',
                vmin=-0.2, vmax=0.2, s=12)
fig.colorbar(sc, ax=ax, fraction=0.05).set_label(r'$r_{\rm SHMR}$ [dex]')
rho_gy = spearmanr(rG['resid'][mm], rY['resid'][mm]).statistic
ax.axhline(0, c='k', lw=0.5, ls='--', alpha=0.4); ax.axvline(0, c='k', lw=0.5, ls='--', alpha=0.4)
ax.set_xlabel(r'$r_{M_{\rm gas}-M}$ [dex]'); ax.set_ylabel(r'$r_{Y-M}$ [dex]')
ax.set_title(rf'$\rho(r_{{M_{{\rm gas}}}}, r_Y) = {rho_gy:+.2f}$ — gas-mass mediated')

# (c) tail-conditional correlations, Truth vs BIND
ax = axes[2]
XKEYS = ['Y-M', 'Mgas-M', 'T-M']
xpos = np.arange(len(XKEYS)); w = 0.19
for k, (src, side, color, hatch) in enumerate([('Truth', '+', 'tomato', None), ('BIND', '+', 'tomato', '//'),
                                               ('Truth', '-', 'steelblue', None), ('BIND', '-', 'steelblue', '//')]):
    vals = [tail_rho(src, 'SHMR', xk, side)[0] for xk in XKEYS]
    ax.bar(xpos + (k - 1.5) * w, vals, w, color=color, hatch=hatch,
           edgecolor='k', lw=0.5, alpha=0.9 if hatch is None else 0.45,
           label=rf'{src} ${side}1\sigma$')
ax.axhline(0, c='k', lw=0.8)
ax.set_xticks(xpos); ax.set_xticklabels([r'$r_{Y-M}$', r'$r_{M_{\rm gas}-M}$', r'$r_{T-M}$'])
ax.set_ylabel(r'$\rho_S$ with $r_{\rm SHMR}$ in tail')
ax.legend(fontsize=8, ncol=2); ax.set_title('tail-conditional correlations')
save_fig(fig, 'figT6d_outlier_physics'); plt.show()


## T7 · Thermo parameter response (`thermo_param.pkl`)

In [ ]:

tpar = L('thermo_param.pkl'); rho = tpar['rho_thermo']['trained']; keys = list(tpar['keys'])
astro = [j for j in range(C.N_PARAMS) if (j+1) not in C.COSMO_PARAM_IDX]
order = sorted(astro); xl=[PARAM_LABELS[j+1] for j in order]
fig, axes = plt.subplots(1, 2, figsize=(15, 3.6), sharey=True)
for ax,(src) in zip(axes, ['True','BIND']):
    d = rho[src][:,order]; im = ax.imshow(d, aspect='auto', cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    ax.set_yticks(range(len(keys))); ax.set_yticklabels(keys); ax.set_xticks(range(len(order))); ax.set_xticklabels(xl, rotation=45, ha='right'); ax.set_title(src)
    for i in range(len(keys)):
        for kk in range(len(order)):
            if np.isfinite(d[i,kk]): ax.text(kk,i,f'{d[i,kk]:.2f}',ha='center',va='center',fontsize=6,color='white' if abs(d[i,kk])>0.45 else 'k')
fig.colorbar(im, ax=axes, fraction=0.012).set_label(r'$\rho_S$')
save_fig(fig, 'figT7_thermo_param_response'); plt.show()


## T8 · Thermo field response (1P butterfly, `field1p.npz` channels 3–6)

In [ ]:

f1 = L('field1p.npz'); params=f1['params']
from scipy.stats import pearsonr
fig, axes = plt.subplots(len(params), 2*len(THK), figsize=(1.9*2*len(THK), 1.9*len(params)))
for ri,j in enumerate(params):
    t_hi,t_lo,g_hi,g_lo = (f1[f'p{j}_{x}'] for x in ('t_hi','t_lo','g_hi','g_lo'))
    if t_hi.shape[0] < 7:  # this 1P sim lacked thermo truth
        for a in axes[ri]: a.axis('off');
        continue
    for c in range(len(THK)):
        eps = 1e-3*np.percentile(np.r_[t_hi[3+c][t_hi[3+c]>0], t_lo[3+c][t_lo[3+c]>0]] if (t_hi[3+c]>0).any() else [1e-6], 99)
        tlr=np.log10((t_hi[3+c]+eps)/(t_lo[3+c]+eps)); glr=np.log10((g_hi[3+c]+eps)/(g_lo[3+c]+eps))
        v=np.quantile(np.abs(np.r_[tlr.ravel(),glr.ravel()]),0.995) or 0.1
        for ki,(fld,kind) in enumerate([(tlr,'Truth'),(glr,'BIND')]):
            ax=axes[ri,c*2+ki]; ax.imshow(fld, origin='lower', cmap='RdBu_r', vmin=-v, vmax=v); ax.set_xticks([]); ax.set_yticks([])
            if ri==0: ax.set_title(f'{THD[THK[c]]}\n{kind}', fontsize=7)
            if c==0 and ki==0: ax.set_ylabel(PARAM_LABELS[int(j)], fontsize=8, rotation=0, ha='right', va='center', labelpad=20)
save_fig(fig, 'figT8_thermo_field_response'); plt.show()
